# Example 01.12 — score with direct REST requests

Performs the same discovery and prediction flow without `ml_app_client`. This is the
low-level equivalent of notebook 01.11 and uses only public REST endpoints.


In [1]:
import getpass
import os
import requests

API = os.getenv("ML_APP_API_URL", "http://localhost:8000/api/v1").rstrip("/")
token = os.getenv("ML_APP_ACCESS_TOKEN", "").strip()
if not token:
    response = requests.post(
        f"{API}/auth/login",
        json={
            "login": input("ML App login or email: "),
            "password": getpass.getpass("ML App password: "),
        },
        timeout=30,
    )
    response.raise_for_status()
    token = response.json()["access_token"]

session = requests.Session()
session.headers.update({"Authorization": f"Bearer {token}", "Accept": "application/json"})
print("Authenticated against", API)


Authenticated against http://localhost:8000/api/v1


In [2]:
from pathlib import Path
import sys

REPOSITORY_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "examples").is_dir()),
    None,
)
if REPOSITORY_ROOT is None:
    raise RuntimeError("Start Jupyter inside the ml-app repository")
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

MODEL_SERVICE_NAME = "Example01 10 - Estates Model Service - demo"
deployments_response = session.get(f"{API}/serving/deployments", timeout=30)
deployments_response.raise_for_status()
matches = [item for item in deployments_response.json() if item["name"] == MODEL_SERVICE_NAME]
if len(matches) != 1:
    raise RuntimeError("Run Example01_10_create_model_service.ipynb first")
service = matches[0]

contract_response = session.get(
    f"{API}/serving/deployments/{service['id']}/input-contract",
    timeout=30,
)
contract_response.raise_for_status()
features = contract_response.json()["example_features"]


In [3]:
response = session.post(
    f"{API}/serving/deployments/{service['id']}/predictions",
    headers={
        "Idempotency-Key": "Example01-rest-score-v2",
        "X-Correlation-ID": "Example01-rest-demo",
    },
    json={"instances": [{
        "record_id": "Example01-rest-record-001",
        "features": features,
    }]},
    timeout=120,
)
response.raise_for_status()
result = response.json()
print({
    "request_id": result["request_id"],
    "model_id": result["model_id"],
    "served_role": result["served_role"],
    "prediction": result["predictions"][0]["prediction"],
    "warnings": result["warnings"],
})


{'request_id': 'a7119bb2-b47c-4dbd-9fca-87483e6578a5', 'model_id': '28c00b2c-4ded-4715-8a19-972168070e7f', 'served_role': 'champion', 'prediction': 1115022.7665175905, 'warnings': []}
